# FastConformer VIVOS Noise Fine-tune - bản local public runner

Notebook này là bản chạy Kaggle gọn cho repo `fci_voice_agent_local`.
Notebook chỉ điều phối; logic chính nằm trong các module nhỏ dưới:

```text
experiments/09_asr_noise_verifier/src/fc_noise_ft/
```

Luồng chạy:

1. Cài dependency.
2. Clone repo `fci_voice_agent_local` từ GitHub public.
3. Thiết lập Hugging Face token nếu model bucket còn private.
4. Tải VIVOS public từ Zenodo và giải nén vào `/kaggle/working`.
5. Import `RunConfig` và `run_pipeline` từ repo.
6. Sinh manifest clean/noisy, eval base, fine-tune, eval sau fine-tune.
7. Xuất scoreboard và artifact.

Yêu cầu trên Kaggle:

- Bật GPU.
- Bật Internet.
- Không cần attach VIVOS thủ công: notebook sẽ tải VIVOS public từ Zenodo.
- Nếu model bucket private, tạo Kaggle Secret tên `HF_TOKEN`.


## 1. Cài dependency

Chạy cell này trước. Nếu Kaggle yêu cầu restart kernel sau khi cài package, restart kernel rồi chạy tiếp từ mục 2.

In [ ]:
%pip install -q --upgrade --no-cache-dir git+https://github.com/huggingface/huggingface_hub.git
%pip install -q jiwer librosa soundfile tqdm omegaconf
%pip install -q "nemo_toolkit[asr]==2.7.3"
%pip install -q --force-reinstall --no-cache-dir "numpy==1.26.4" "scipy==1.12.0" "pandas==2.2.2"
%pip uninstall -y numba-cuda

print("Install done. Restart the Kaggle kernel now, then continue from section 2.")

## 2. Clone repo local public

Sau khi bạn push bản local lên GitHub public, Kaggle sẽ clone repo này. Nếu đổi tên repo, chỉ cần sửa `REPO_URL`.

In [ ]:
REPO_URL = "https://github.com/ManhTanTran/fci_voice_agent_local.git"
REPO_DIR = "/kaggle/working/fci_voice_agent_local"

!rm -rf "$REPO_DIR"
!git clone --depth 1 "$REPO_URL" "$REPO_DIR"
%cd /kaggle/working/fci_voice_agent_local/experiments/09_asr_noise_verifier
!git log -1 --oneline

## 3. Thiết lập Hugging Face token

Cell này chỉ cần khi model `.nemo` nằm trong bucket/repo private. Nếu model đã public hoặc được upload thành Kaggle Dataset, có thể bỏ qua.

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as exc:
    print("No Kaggle HF_TOKEN secret or cannot read it:", repr(exc))

## 4. Tải VIVOS public

Cell này tải VIVOS public từ Zenodo, giải nén vào `/kaggle/working/vivos_raw`, rồi kiểm tra có `prompts.txt` trước khi chạy pipeline.

In [ ]:
VIVOS_URL = "https://zenodo.org/records/7068130/files/vivos.tar.gz?download=1"
VIVOS_TAR = "/kaggle/working/vivos.tar.gz"
VIVOS_DIR = "/kaggle/working/vivos_raw"

!wget -O "$VIVOS_TAR" "$VIVOS_URL"
!mkdir -p "$VIVOS_DIR"
!tar -xzf "$VIVOS_TAR" -C "$VIVOS_DIR"

matches = list(Path(VIVOS_DIR).rglob("prompts.txt"))
print(matches[:20])
print("n_prompts:", len(matches))

## 5. Import module từ repo

Notebook không chứa logic train/eval dài. Toàn bộ logic nằm trong package `fc_noise_ft` để dễ sửa bằng GitHub.

In [ ]:
import json
import sys
from pathlib import Path

SRC_DIR = Path.cwd() / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from fc_noise_ft.config import RunConfig
from fc_noise_ft.pipeline import run_pipeline

print("Loaded modules from", SRC_DIR)

## 6. Cấu hình run

Mặc định giữ subset nhỏ để smoke test. Khi luồng đã chạy ổn, tăng `MAX_TRAIN_BASE`, `MAX_DEV_BASE`, `MAX_TEST_BASE`, `MAX_EPOCHS`.

In [ ]:
cfg = RunConfig(
    vivos_root=VIVOS_DIR,
    work_dir="/kaggle/working/noise_ft",
    model_uri="hf://buckets/MarkTT1/vi-asr-fastconformer-114m-bucket/s3-fc115m-full.nemo",
    model_path="/kaggle/working/models/s3-fc115m-full.nemo",
    max_train_base=800,
    max_dev_base=120,
    max_test_base=200,
    max_epochs=1,
    batch_size=4,
    accum=4,
    eval_batch_size=8,
    lr=1e-4,
    skip_train=False,
    skip_base_eval=False,
)

Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
print(json.dumps(cfg.to_dict(), ensure_ascii=False, indent=2, default=str))

## 7. Chạy pipeline

Cell này chạy toàn bộ pipeline:

- tải model `.nemo`;
- scan VIVOS từ thư mục vừa tải;
- split train/dev/test;
- sinh clean/noisy manifest;
- eval base model;
- fine-tune;
- eval model sau fine-tune;
- ghi scoreboard CSV/JSON.

In [ ]:
results = run_pipeline(cfg)
print(json.dumps(results, ensure_ascii=False, indent=2, default=str)[:4000])

## 8. Xem scoreboard

Bảng chính là `wer_base_vs_finetuned.csv`. `delta_abs < 0` nghĩa là fine-tuned tốt hơn base theo WER tuyệt đối.

In [ ]:
import pandas as pd
from IPython.display import display

work = Path(cfg.work_dir)
summary_path = work / "wer_base_vs_finetuned.csv"
base_path = work / "wer_base.csv"
ft_path = work / "wer_finetuned.csv"

if summary_path.exists():
    df = pd.read_csv(summary_path)
    display(df)
else:
    print("Missing summary:", summary_path)
    if base_path.exists():
        print("Base WER:")
        display(pd.read_csv(base_path))
    if ft_path.exists():
        print("Fine-tuned WER:")
        display(pd.read_csv(ft_path))

## 9. File output cần giữ

```text
/kaggle/working/noise_ft/results.json
/kaggle/working/noise_ft/wer_base.csv
/kaggle/working/noise_ft/wer_finetuned.csv
/kaggle/working/noise_ft/wer_base_vs_finetuned.csv
/kaggle/working/noise_ft/pred_base.csv
/kaggle/working/noise_ft/pred_finetuned.csv
/kaggle/working/noise_ft/fastconformer_noisy_ft.nemo
```

Sau khi có kết quả ổn từ bản local, mới copy scoreboard/model note sang repo chính theo quy tắc bạn đang đặt.